In [ ]:
%%bash

sudo apt install libopenblas-dev -y
pip install jupyterplot
pip install git+https://github.com/lbaitemple/Proto-Grid.git@dev
pip install  bokeh jupyterlab-executor
pip install bokeh --upgrade


In [ ]:
!pip show revidyne

In [ ]:
!mkdir -p streaming

In [ ]:
%%writefile streaming/main.py

from revidyne import AllDevice

from bokeh import models, plotting, io
from bokeh.io import show
from bokeh.layouts import column
import pandas as pd
from itertools import cycle
from datetime import datetime
import time
import math

a=AllDevice(115200)

mm=a.getAllDevice()
g0 = mm['generator0'].getDevice()
h0 = mm['houseload0'].getDevice()

# Settle out after initial setup
time.sleep(10)

# Set initial parameters - generator 0
g0.call('trackOn')
gen0State = 'ON'
g0.setLoad(150)
g0.setKp(5)
g0.setKd(1.5)
g0.setKi(2)

nowTime = datetime.now().strftime("%H:%M")
outLoad = float(g0.call('getKW')[0])
outLoad = 0 if math.isnan(outLoad) else outLoad
i = 0
source = models.ColumnDataSource(data=dict(Time=[i], Output=[outLoad]))

def toggle_gen0Track():
    global gen0State
    if gen0State == 'ON':
        g0.call('trackOff')
        gen0State = 'OFF'
    else:
        g0.call('trackOn')
        gen0State = 'ON'

def setGen0Kp(attr, old, new):
    g0.setKp(new)

def setGen0Ki(attr, old, new):
    g0.setKi(new)

def setGen0Kd(attr, old, new):
    g0.setKd(new)

def update(event=None):
    time.sleep(1)
    nowTime = datetime.now().strftime("%H:%M")
    global i
    i = i + 1
    data = g0.call('getKW')[0]
    if (data != ''):
        outLoad = float(data)
    else:
        outLoad =0
        
    outLoad = 0 if math.isnan(outLoad) else outLoad
    new_data={'Time': [i], 'Output': [outLoad]}
    source.stream(new_data)
    # print(now.strftime("%H:%M"), g1.call('getKW'))

p = plotting.figure(
    x_axis_label="Time", y_axis_label="Output",
    width=800, height=400, x_axis_type="linear", # , tools=["hover", "wheel_zoom"]
)
p.line(x="Time", y="Output", source=source, width=4)

io.curdoc().add_root(p)

button = models.widgets.Button(label='Generator0 Track ON/OFF')
button.on_click(toggle_gen0Track)

Gen0Kp = models.widgets.Slider(start=0.1, end=10, value=1, step=0.1, title="Generator0 Kp")
Gen0Kd = models.widgets.Slider(start=0.1, end=5, value=1, step=0.1, title="Generator0 Kd")
Gen0Ki = models.widgets.Slider(start=0.1, end=5, value=1, step=0.1, title="Generator0 Ki")

Gen0Kp.on_change('value', setGen0Kp)
Gen0Ki.on_change('value', setGen0Ki)
Gen0Kd.on_change('value', setGen0Kd)

io.curdoc().add_root(button)
io.curdoc().add_root(Gen0Kp)
io.curdoc().add_root(Gen0Kd)
io.curdoc().add_root(Gen0Ki)

io.curdoc().add_periodic_callback(update, 10)

In [13]:
import psutil, os

for interface, addrs in psutil.net_if_addrs().items():
    for addr in addrs:
        if addr.family == socket.AF_INET and not addr.address.startswith("127."):
            print(f"Interface: {interface}, LAN IP: {addr.address}")
            os.environ["IP"] = addr.address
            break

Interface: wlan0, LAN IP: 192.168.86.27


In [14]:
!echo $IP

192.168.86.27


In [ ]:
!echo $HOSTIP

In [ ]:
!python3 -m bokeh  serve streaming --allow-websocket-origin='*' --address $IP --port 5006

2025-03-25 19:46:31,121 Starting Bokeh server version 3.4.3 (running on Tornado 6.4.2)
2025-03-25 19:46:31,123 Host wildcard '*' will allow connections originating from multiple (or possibly all) hostnames or IPs. Use non-wildcard values to restrict access explicitly
2025-03-25 19:46:31,125 User authentication hooks NOT provided (default user enabled)
2025-03-25 19:46:31,132 Bokeh app running at: http://192.168.86.27:5006/streaming
2025-03-25 19:46:31,133 Starting Bokeh server with process id: 15909
/dev/ttyUSB1
/dev/ttyUSB0
['/dev/ttyUSB1', '/dev/ttyUSB0']
Disconnected from /dev/ttyUSB1.
Disconnected from /dev/ttyUSB0.
/dev/ttyUSB1
/dev/ttyUSB0
generator
generator0
generator0
/dev/ttyUSB1
/dev/ttyUSB0
['/dev/ttyUSB1', '/dev/ttyUSB0']
Disconnected from /dev/ttyUSB1.
Disconnected from /dev/ttyUSB0.
houseload
houseload0
houseload0
2025-03-25 19:47:11,204 WebSocket connection opened
2025-03-25 19:47:12,308 ServerConnection created


In [ ]:
!pip install bokeh --upgrade